# exp123_cross_test_prefix_label_context_audit train

同じ pseudo test batch 内の他 well に見えている `TVT_input` prefix label から、batch-level の bias / slope / residual scale を診断する。提出生成は行わない。

## Contents

1. Setup and configuration
2. Input preview
3. Run cross-test prefix label audit
4. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd

from cross_test_prefix_label_context_audit import run_audit, write_metrics
from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_WELLS_ENV = os.environ.get("EXPERIMENT_MAX_WELLS")
MAX_WELLS = int(MAX_WELLS_ENV) if MAX_WELLS_ENV else (40 if DEBUG else None)

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", EXPERIMENT_NAME)
print("Route:", get_nested(config, "experiment.route"))
print("Parent:", get_nested(config, "lineage.parent"))
print("Train data:", paths.train_data_dir)
print("Artifacts:", paths.artifacts_dir)
print("Debug:", DEBUG, "Max wells:", MAX_WELLS)
print("Rules-risk diagnostic:", get_nested(config, "validation.leakage_policy"))

## 2. Input preview


In [ ]:
train_files = sorted(paths.train_data_dir.glob("*__horizontal_well.csv"))
print("horizontal train files:", len(train_files))
if not train_files:
    raise FileNotFoundError(f"No horizontal train files found under {paths.train_data_dir}")

preview_path = train_files[0]
preview = pd.read_csv(preview_path, nrows=8)
print("preview path:", preview_path)
print("columns:", list(preview.columns))
display(preview[["MD", "TVT", "TVT_input"]])

## 3. Run cross-test prefix label audit


In [ ]:
summary = run_audit(paths=paths, config=config, output_dir=paths.artifacts_dir, max_wells=MAX_WELLS)
write_metrics(paths, summary)
print(json.dumps(summary, indent=2, ensure_ascii=False)[:4000])
print("metrics path:", paths.metrics_path)

## 4. Metrics and artifacts


In [ ]:
candidate_metrics = pd.read_csv(paths.artifacts_dir / "cross_test_prefix_label_candidate_metrics.csv")
bucket_metrics = pd.read_csv(paths.artifacts_dir / "cross_test_prefix_label_bucket_metrics.csv")
context_stats = pd.read_csv(paths.artifacts_dir / "cross_test_prefix_label_context_stats.csv")

display(candidate_metrics)
display(bucket_metrics.head(24))
display(context_stats.head())

print("artifact files:")
for path in sorted(paths.artifacts_dir.glob("cross_test_prefix_label_*")):
    print("-", path.name, path.stat().st_size)